# 🦙 Medical Q&A Fine-Tuning with LoRA + TinyLlama-1.1B

**Author:** Sonal Mishra  
**Model:** `TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T`  
**Dataset:** [medalpaca/medical_meadow_medqa](https://huggingface.co/datasets/medalpaca/medical_meadow_medqa)  
**Method:** PEFT — Parameter-Efficient Fine-Tuning via **LoRA** (Low-Rank Adaptation)  
**Platform:** Google Colab (T4 GPU)

---

## 📌 What This Notebook Covers

This notebook demonstrates fine-tuning a 1.1B parameter LLM for **medical question answering** using LoRA — a parameter-efficient method that updates only ~0.5% of model weights while preserving the full pre-trained knowledge.

> **Core Idea:** Instead of updating all 1.1 billion weights (expensive), LoRA injects small trainable matrices into attention layers. The original weights are frozen. Only these tiny adapters are trained.

### Why LoRA over Full Fine-Tuning?

| | Full Fine-Tuning | LoRA |
|---|---|---|
| Trainable params | 100% (1.1B) | ~0.5% (~6M) |
| GPU memory | ~16GB+ | ~6–8GB |
| Training time | Hours | Minutes |
| Accuracy loss | None | Minimal |
| Portability | Full model copy | Small adapter file |

### Pipeline at a Glance
```
TinyLlama-1.1B (frozen base weights)
        ↓
Inject LoRA Adapters into q_proj + v_proj
        ↓
Load Medical QA Dataset (MedAlpaca)
        ↓
Format → Alpaca Prompt Template
        ↓
Tokenize (causal LM: labels = input_ids)
        ↓
Train LoRA Adapters Only (3 epochs)
        ↓
Compare Fine-Tuned vs. Base Model
```


---
## Section 1: Install Dependencies

Beyond the standard HuggingFace stack, we add two new libraries:

- **`peft`** — HuggingFace's PEFT library that implements LoRA, prefix tuning, adapters, etc.
- **`bitsandbytes`** — enables quantization (int8/int4), critical for running large models on free Colab GPUs


In [1]:
# Install HuggingFace ecosystem + PEFT + bitsandbytes for quantization
!pip install transformers peft datasets accelerate bitsandbytes -q

---
## Section 2: Load TinyLlama-1.1B Model & Tokenizer

**TinyLlama** is a compact 1.1B parameter LLM trained on 3 trillion tokens — following the LLaMA-2 architecture.  
Despite its small size, it demonstrates strong reasoning capabilities and is an ideal base for fine-tuning experiments on limited compute.

### Key Loading Decisions

**`torch_dtype=torch.float16`**  
Uses 16-bit floating point instead of the default 32-bit. This halves memory usage with negligible precision loss.  
> **Analogy:** You can describe the height of a person as "5 feet 11.234567 inches" (float32) or "5 feet 11 inches" (float16). For most purposes, the extra decimals don't matter — and float16 fits two numbers in the space of one.

**`device_map="auto"`**  
HuggingFace automatically distributes model layers across available GPU/CPU memory.  
On a single Colab T4 (16GB), the entire model fits on GPU. On smaller setups, it splits across GPU + CPU.

**`tokenizer.pad_token = tokenizer.eos_token`**  
TinyLlama's tokenizer doesn't have a dedicated pad token (common in decoder-only LLMs).  
We reuse the end-of-sequence token for padding — this is standard practice for causal LMs.


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# ── Load Tokenizer ─────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Decoder-only LLMs don't have a dedicated pad token by default
# We assign EOS token as the pad token — standard practice for causal LMs
tokenizer.pad_token = tokenizer.eos_token

# ── Load Model ─────────────────────────────────────────────────────────────────
# torch_dtype=float16: halves memory usage (1.1B params × 2 bytes ≈ 2.2GB)
# device_map="auto": automatically places layers on GPU/CPU based on available VRAM
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("✅ Model loaded successfully!")
print(f"   Architecture: {model.config.model_type}")
print(f"   Hidden size:  {model.config.hidden_size}")
print(f"   Num layers:   {model.config.num_hidden_layers}")
print(f"   Num heads:    {model.config.num_attention_heads}")
print()
print(model)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model loaded successfully!
   Architecture: llama
   Hidden size:  2048
   Num layers:   22
   Num heads:    32

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attenti

---
## Section 3: Configure and Apply LoRA Adapters

This is the heart of the notebook. LoRA works by inserting small trainable matrices into the model's attention layers, while keeping the original weights completely frozen.

### How LoRA Works

For a pre-trained weight matrix **W** (e.g., the query projection matrix in an attention layer):

Instead of updating **W** directly, LoRA learns two small matrices:
- **A** — shape `[r × d_in]`  (randomly initialized)
- **B** — shape `[d_out × r]`  (initialized to zero, so initial output = 0)

The effective weight becomes: **W + B×A × (lora_alpha/r)**

> **Analogy:** Imagine a senior consultant (the frozen base model) who has deep expertise. Instead of retraining them from scratch, you add a junior assistant (the LoRA adapter) who learns task-specific adjustments and whispers corrections to the consultant's answers. The consultant's core knowledge is preserved; only the assistant learns.

### LoRA Hyperparameters Explained

**`r=8` (rank)**  
The bottleneck dimension of the adapter matrices. A rank-8 adapter for a 1024×1024 weight matrix means: instead of learning 1M parameters, you learn only 8×1024 + 8×1024 = 16K parameters.  
- Higher r → more expressive but more parameters
- r=4 to r=16 is typical for instruction fine-tuning

**`lora_alpha=16` (scaling factor)**  
Controls how much the adapter output is scaled before being added to the frozen weights.  
The effective scale = `lora_alpha / r` = 16/8 = **2.0**  
Think of it as the volume knob for how loudly the adapter speaks relative to the base model.

**`target_modules=["q_proj", "v_proj"]`**  
We only apply LoRA to the **Query** and **Value** projection matrices in each attention layer.  
These are empirically found to be the most impactful layers for task adaptation.  
Applying LoRA to all linear layers (q, k, v, o, gate, up, down projections) would be more thorough but use more memory.

**`lora_dropout=0.05`**  
Small dropout applied to adapter outputs during training to prevent overfitting.

**`task_type="CAUSAL_LM"`**  
Tells PEFT this is an auto-regressive language model (generates text token by token, left to right).


In [3]:
from peft import LoraConfig, get_peft_model

# ── Define LoRA Configuration ──────────────────────────────────────────────────
config = LoraConfig(
    r=8,                                   # Rank: bottleneck dimension of adapter matrices
    lora_alpha=16,                         # Scaling factor: effective scale = alpha/r = 2.0
    target_modules=["q_proj", "v_proj"],   # Apply LoRA to Query and Value projections
    lora_dropout=0.05,                     # Dropout on adapter outputs (regularization)
    bias="none",                           # Don't add trainable bias terms
    task_type="CAUSAL_LM"                  # Auto-regressive text generation task
)

# ── Wrap Base Model with LoRA ──────────────────────────────────────────────────
# This freezes all base model weights and injects LoRA adapter matrices
model = get_peft_model(model, config)

# ── Verify Trainable Parameters ───────────────────────────────────────────────
# Expected: ~0.5% of total parameters are trainable
model.print_trainable_parameters()

# Breakdown: for each attention layer, we add:
#   q_proj adapter: [r × d_hidden] + [d_hidden × r]
#   v_proj adapter: [r × d_hidden] + [d_hidden × r]
#   With d_hidden=2048, r=8: 2×2×(8×2048) = ~65K per layer × 22 layers ≈ 1.4M params


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


---
## Section 4: Load the Medical QA Dataset

**MedAlpaca's medical_meadow_medqa** contains medical multiple-choice questions derived from the US Medical Licensing Examination (USMLE) question bank.

Each sample has three fields:
- `instruction` — the task description ("Please answer with one of the option in the bracket")
- `input` — the clinical question + answer choices
- `output` — the correct answer

This is the Alpaca format — a widely used instruction fine-tuning template.


In [4]:
from datasets import load_dataset

# ── Load MedAlpaca Medical QA Dataset ─────────────────────────────────────────
dataset = load_dataset("medalpaca/medical_meadow_medqa", split="train")

print("📦 Dataset Overview:")
print(dataset)

print("\n🔍 Sample Example:")
print(f"  Instruction : {dataset[0]['instruction'][:100]}...")
print(f"  Input       : {dataset[0]['input'][:150]}...")
print(f"  Output      : {dataset[0]['output']}")


📦 Dataset Overview:
Dataset({
    features: ['input', 'instruction', 'output'],
    num_rows: 10178
})

🔍 Sample Example:
  Instruction : Please answer with one of the option in the bracket...
  Input       : Q:A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening desp...
  Output      : E: Nitrofurantoin


---
## Section 5: Format Data Using the Alpaca Prompt Template

We structure each example into a consistent 3-part prompt format:
```
### Instruction:
<task description>

### Input:
<clinical question + answer choices>

### Response:
<correct answer>
```

### Why This Template Matters

The model must learn to:
1. Recognize `### Instruction:` as the task context
2. Recognize `### Input:` as the question
3. **Generate** the correct answer after `### Response:`

During inference, we provide everything up to `### Response:` and let the model complete it.

> **Analogy:** This is like training a student with structured flash cards. Every card has the same layout — task header, question, then answer. After seeing thousands of these, the student learns: "when I see `### Response:`, it's my turn to answer."

Using a consistent template is critical for instruction fine-tuning — inconsistent formatting confuses the model and degrades performance significantly.


In [5]:
def format_prompt(row):
    """
    Format a dataset row into the Alpaca instruction-following template.

    The three-part structure (Instruction / Input / Response) teaches the model
    the consistent pattern: read the task → read the question → produce an answer.

    Args:
        row (dict): dataset row with 'instruction', 'input', 'output' fields

    Returns:
        dict: row with additional 'text' field containing the formatted prompt
    """
    text = f"""### Instruction:
{row['instruction']}

### Input:
{row['input']}

### Response:
{row['output']}"""
    return {"text": text}

# ── Apply formatting to entire dataset ────────────────────────────────────────
dataset = dataset.map(format_prompt)

print("✅ Formatting applied!")
print("\n📄 Sample formatted prompt:")
print("-" * 60)
print(dataset[0]["text"])
print("-" * 60)


✅ Formatting applied!

📄 Sample formatted prompt:
------------------------------------------------------------
### Instruction:
Please answer with one of the option in the bracket

### Input:
Q:A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?? 
{'A': 'Ampicillin', 'B': 'Ceftriaxone', 'C': 'Ciprofloxacin', 'D': 'Doxycycline', 'E': 'Nitrofurantoin'},

### Response:
E: Nitrofurantoin
------------------------------------------------------------


---
## Section 6: Tokenize for Causal Language Modelling

### The Critical Difference from Classification

In the DistilBERT fine-tuning notebook, we set labels separately (0 or 1 for sentiment).  
For causal language modelling, **labels = input_ids** — the model's job is to predict each next token from the previous tokens.

This means the loss is computed across the entire sequence — the model learns to predict:
- `### Input:` after `### Instruction:`
- Each word in the question
- Most importantly: the **correct answer after `### Response:`**

> **Analogy:** Teaching someone to fill in a story by providing the full story during training. They learn every transition — but the test is whether they can complete a story from just the beginning.

### Why `remove_columns=dataset.column_names`?

After tokenization, the raw text columns (`instruction`, `input`, `output`, `text`) are no longer needed — the model only uses `input_ids`, `attention_mask`, and `labels`. Removing them reduces memory usage.


In [6]:
def tokenize(row):
    """
    Tokenize a formatted prompt for causal language modelling.

    Key difference from classification:
    - labels = input_ids  (model predicts each next token)
    - The loss is computed across ALL tokens, not just a final label
    - This teaches the model the full Instruction → Input → Response pattern

    Args:
        row (dict): row with 'text' field (formatted Alpaca prompt)

    Returns:
        dict: input_ids, attention_mask, labels tensors
    """
    result = tokenizer(
        row["text"],
        truncation=True,       # Trim sequences longer than max_length
        max_length=512,        # Medical questions can be lengthy; 512 covers most
        padding="max_length"   # Pad shorter sequences for uniform batch shape
    )

    # For causal LM: the model predicts the next token at every position
    # So labels are simply a copy of input_ids (shifted by 1 internally by HuggingFace)
    result["labels"] = result["input_ids"].copy()

    return result

# ── Tokenize entire dataset ────────────────────────────────────────────────────
# remove_columns: drop raw text columns, keep only tensor fields
tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

print("✅ Tokenization complete!")
print(tokenized_dataset)
print(f"\n   Columns: {tokenized_dataset.column_names}")
print(f"   Input IDs shape per sample: {len(tokenized_dataset[0]['input_ids'])} tokens")


✅ Tokenization complete!
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 10178
})

   Columns: ['input_ids', 'attention_mask', 'labels']
   Input IDs shape per sample: 512 tokens


---
## Section 7: Configure Training Arguments & Train

### Key Hyperparameter Decisions

**`learning_rate=2e-4`**  
Notice this is 10× higher than the 2e-5 we used for DistilBERT full fine-tuning.  
This is intentional: since the base model weights are **frozen**, we can't cause catastrophic forgetting.  
Only the tiny LoRA adapters are being trained, and they start from zero — so they need a higher learning rate to learn effectively.

**`gradient_accumulation_steps=4`**  
With batch_size=4, this effectively simulates a batch of 4×4=16 before each weight update.  
This is a memory trick: instead of loading 16 samples at once (which might OOM), we process 4 samples at a time and accumulate gradients across 4 steps before updating.

**`warmup_steps=50`**  
The learning rate starts near zero and linearly ramps up over 50 steps, then follows the scheduler.  
This prevents large, unstable updates in early training when the LoRA adapters are far from optimal.

**`fp16=True`**  
Enables mixed precision training — forward pass in float16 (fast), gradient accumulation in float32 (stable).  
This halves GPU memory usage during training without sacrificing convergence.

**`DataCollatorForLanguageModeling(mlm=False)`**  
`mlm=False` tells the collator this is causal LM (not masked LM like BERT).  
It handles left-to-right text generation batching correctly.


In [7]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# ── Training Configuration ─────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./tinyllama-medical-lora",   # Checkpoint and adapter save directory
    num_train_epochs=3,                       # 3 epochs: sufficient for task adaptation on MedAlpaca
    per_device_train_batch_size=4,            # Samples per GPU step
    gradient_accumulation_steps=4,           # Accumulate 4 steps → effective batch = 16
    warmup_steps=50,                          # Gradual LR ramp-up for stable early training
    learning_rate=2e-4,                       # 10× higher than full fine-tuning — safe since base is frozen
    fp16=True,                                # Mixed precision: float16 forward, float32 gradients
    logging_steps=50,                         # Log training loss every 50 steps
    save_steps=500,                           # Save checkpoint every 500 steps
    report_to="none",                         # Disable W&B / MLflow logging
)

# ── Data Collator ─────────────────────────────────────────────────────────────
# mlm=False → causal language modelling (predict next token)
# mlm=True  → masked language modelling (predict masked tokens, BERT-style)
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# ── Initialize Trainer ─────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# ── Start Training ─────────────────────────────────────────────────────────────
print("🚀 Starting LoRA fine-tuning...")
print(f"   Base model weights  : FROZEN")
print(f"   Trainable params    : LoRA adapters only (~0.5% of total)")
print(f"   Effective batch size: {4 * 4} (batch × grad_accum)")
print()

trainer.train()


🚀 Starting LoRA fine-tuning...
   Base model weights  : FROZEN
   Trainable params    : LoRA adapters only (~0.5% of total)
   Effective batch size: 16 (batch × grad_accum)



Step,Training Loss
50,1.637725
100,1.231325
150,1.187103
200,1.174137
250,1.165547
300,1.155800
350,1.149293
400,1.160998
450,1.141115
500,1.147499


TrainOutput(global_step=1911, training_loss=1.130670552475798, metrics={'train_runtime': 5350.5037, 'train_samples_per_second': 5.707, 'train_steps_per_second': 0.357, 'total_flos': 9.714338190537523e+16, 'train_loss': 1.130670552475798, 'epoch': 3.0})

---
## Section 8: Save the Fine-Tuned LoRA Adapter

An important distinction from full fine-tuning: **`model.save_pretrained()` on a LoRA model saves only the adapter weights — not the full model.**

This means the saved files are tiny (a few MB instead of ~2GB for the full model).  
To deploy, you load the base TinyLlama model and merge/load the adapter on top.


In [8]:
SAVE_PATH = "./tinyllama-medical-lora"

# ── Save LoRA Adapter Weights ──────────────────────────────────────────────────
# On a PEFT model, save_pretrained saves ONLY the adapter weights (not the full base model)
# This produces a much smaller output (~few MB vs ~2GB for full model)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"✅ LoRA adapter and tokenizer saved to: {SAVE_PATH}")
print("\nSaved files:")
import os
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(f"{SAVE_PATH}/{f}") / 1024
    print(f"  📄 {f} ({size:.1f} KB)")


✅ LoRA adapter and tokenizer saved to: ./tinyllama-medical-lora

Saved files:
  📄 tokenizer_config.json (0.3 KB)
  📄 tokenizer.json (3534.2 KB)
  📄 adapter_model.safetensors (4411.3 KB)
  📄 README.md (5.1 KB)
  📄 checkpoint-1000 (4.0 KB)
  📄 adapter_config.json (1.0 KB)
  📄 checkpoint-1500 (4.0 KB)
  📄 checkpoint-1911 (4.0 KB)
  📄 checkpoint-637 (4.0 KB)
  📄 checkpoint-500 (4.0 KB)


---
## Section 9: Inference — Generate Medical Answers

We define a `generate()` function that:
1. Formats the prompt with the Alpaca template
2. Tokenizes and sends to GPU
3. Generates the response using **greedy decoding** (`temperature=0, do_sample=False`)
4. Extracts only the part after `### Response:` for clean output

### Why `temperature=0` for Evaluation?

`temperature=0` means the model **always picks the highest probability token** — completely deterministic.  
This is equivalent to argmax decoding (greedy search).  
For medical QA evaluation, we want consistency — the same question should always give the same answer.

> **Analogy:** A doctor choosing a diagnosis shouldn't randomly pick from their top 3 guesses each time they see the same patient. `temperature=0` enforces the model to commit to its best answer consistently.


In [9]:
def generate(model, prompt, max_new_tokens=100):
    """
    Generate a response from a formatted Alpaca prompt.

    Args:
        model: Fine-tuned or base TinyLlama model
        prompt (str): Alpaca-formatted prompt (up to ### Response:)
        max_new_tokens (int): maximum new tokens to generate

    Returns:
        str: extracted response (text after ### Response:)
    """
    # Tokenize and send to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0,      # Greedy decoding: always pick the highest probability token
            do_sample=False,    # No sampling — fully deterministic output
        )

    # Decode the full output (input + generated tokens)
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the response portion (everything after "### Response:")
    if "### Response:" in full_output:
        response = full_output.split("### Response:")[-1].strip()
        return response.split("\n")[0].strip()   # Return only the first line of the response
    return full_output


# ── Test: Single Inference Example ────────────────────────────────────────────
prompt = """### Instruction:
Please answer with one of the option in the bracket

### Input:
Q: A 45-year-old man presents with chest pain radiating to his left arm and sweating. ECG shows ST elevation in leads II, III, aVF.
{'A': 'Anterior MI', 'B': 'Inferior MI', 'C': 'Lateral MI', 'D': 'Posterior MI'}

### Response:"""

print("=== FINE-TUNED MODEL ===")
print(generate(model, prompt))
# Expected: B (Inferior MI — ST elevation in II, III, aVF is classic inferior STEMI)


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== FINE-TUNED MODEL ===
A: Anterior MI


---
## Section 10: Load Base Model — Baseline Comparison

To fairly compare, we load the **base TinyLlama** (without LoRA) and run the same prompts.

Note: We load `TinyLlama-1.1B-Chat-v1.0` (the instruction-tuned chat version) as the baseline — this is a stronger baseline than the raw pretrained model, ensuring our comparison is fair.

If the fine-tuned model outperforms even the chat-tuned baseline on medical questions, that validates our LoRA adapter.


In [10]:
from transformers import AutoModelForCausalLM

# ── Load Base TinyLlama (Chat version — strong baseline) ──────────────────────
base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto"
)

print("✅ Base model loaded for comparison")
print("\n=== BASE MODEL PREDICTION (no medical fine-tuning) ===")
print(generate(base_model, prompt))


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Base model loaded for comparison

=== BASE MODEL PREDICTION (no medical fine-tuning) ===
Based on the ECG, the patient has an anterior MI.


---
## Section 11: Full Benchmark — Fine-Tuned vs. Base Model on 3 Medical Cases

We test on three distinct medical question types:
1. **Cardiology** — ECG reading (STEMI localisation)
2. **Pharmacology** — Drug safety in pregnancy
3. **Neurology** — Classic clinical presentation (meningitis triad)


In [11]:
test_prompts = [
    # Case 1: ECG interpretation — Inferior STEMI
    """### Instruction:
Please answer with one of the option in the bracket

### Input:
Q: A 45-year-old man presents with chest pain and ST elevation in leads II, III, aVF.
{'A': 'Anterior MI', 'B': 'Inferior MI', 'C': 'Lateral MI', 'D': 'Posterior MI'}

### Response:""",

    # Case 2: Pharmacology — antibiotic safety in pregnancy
    """### Instruction:
Please answer with one of the option in the bracket

### Input:
Q: Which antibiotic is safe in pregnancy for UTI treatment?
{'A': 'Ciprofloxacin', 'B': 'Doxycycline', 'C': 'Nitrofurantoin', 'D': 'Tetracycline'}

### Response:""",

    # Case 3: Neurology — classic meningitis triad (fever + neck stiffness + photophobia)
    """### Instruction:
Please answer with one of the option in the bracket

### Input:
Q: A patient has high fever, neck stiffness and photophobia. What is the diagnosis?
{'A': 'Migraine', 'B': 'Meningitis', 'C': 'Encephalitis', 'D': 'Brain tumor'}

### Response:""",
]

# Expected answers: B (Inferior MI), C (Nitrofurantoin), B (Meningitis)

print("=" * 55)
print("     FINE-TUNED MODEL (LoRA + Medical QA Training)")
print("=" * 55)
for i, prompt in enumerate(test_prompts):
    print(f"  Q{i+1}: {generate(model, prompt)}")

print()
print("=" * 55)
print("     BASE MODEL (No Medical Fine-Tuning)")
print("=" * 55)
for i, prompt in enumerate(test_prompts):
    print(f"  Q{i+1}: {generate(base_model, prompt)}")


     FINE-TUNED MODEL (LoRA + Medical QA Training)
  Q1: A: Anterior MI
  Q2: C: Nitrofurantoin
  Q3: A: Migraine

     BASE MODEL (No Medical Fine-Tuning)
  Q1: Based on the patient's chest pain and ST elevation in leads II, III, aVF, the most likely diagnosis is an anterior MI.
  Q2: C: Ciprofloxacin is safe in pregnancy for UTI treatment.
  Q3: Based on the symptoms, the diagnosis is 'Meningitis'


In [12]:
# ── Evaluation with Correct Answer Comparison ────────────────────────────────

test_cases = [
    {
        "question": "ECG shows ST elevation in leads II, III, aVF. Diagnosis?",
        "choices": "A: Anterior MI | B: Inferior MI | C: Lateral MI | D: Posterior MI",
        "correct": "B",
        "correct_label": "B: Inferior MI",
        "prompt": """### Instruction:
Please answer with one of the option in the bracket

### Input:
Q: A 45-year-old man presents with chest pain and ST elevation in leads II, III, aVF.
{'A': 'Anterior MI', 'B': 'Inferior MI', 'C': 'Lateral MI', 'D': 'Posterior MI'}

### Response:"""
    },
    {
        "question": "Which antibiotic is safe in pregnancy for UTI treatment?",
        "choices": "A: Ciprofloxacin | B: Doxycycline | C: Nitrofurantoin | D: Tetracycline",
        "correct": "C",
        "correct_label": "C: Nitrofurantoin",
        "prompt": """### Instruction:
Please answer with one of the option in the bracket

### Input:
Q: Which antibiotic is safe in pregnancy for UTI treatment?
{'A': 'Ciprofloxacin', 'B': 'Doxycycline', 'C': 'Nitrofurantoin', 'D': 'Tetracycline'}

### Response:"""
    },
    {
        "question": "High fever + neck stiffness + photophobia. Diagnosis?",
        "choices": "A: Migraine | B: Meningitis | C: Encephalitis | D: Brain tumor",
        "correct": "B",
        "correct_label": "B: Meningitis",
        "prompt": """### Instruction:
Please answer with one of the option in the bracket

### Input:
Q: A patient has high fever, neck stiffness and photophobia. What is the diagnosis?
{'A': 'Migraine', 'B': 'Meningitis', 'C': 'Encephalitis', 'D': 'Brain tumor'}

### Response:"""
    },
]


def extract_letter(response):
    """
    Extract the answer letter (A/B/C/D) from a model response.
    Handles formats like 'B', 'B:', 'B: Inferior MI', or verbose sentences.
    """
    response = response.strip()
    # Check if response starts with a letter followed by colon or space
    if response and response[0].upper() in "ABCD":
        return response[0].upper()
    # Scan for option letters mentioned in the response
    for letter in ["A", "B", "C", "D"]:
        if f"{letter}:" in response or f"option {letter}" in response.lower():
            return letter
    return "?"   # Could not parse


def evaluate_model(model, test_cases, model_label):
    """
    Run model on all test cases and print a results table with ✅/❌ per question.

    Args:
        model: HuggingFace model (fine-tuned or base)
        test_cases (list): list of dicts with prompt, correct, correct_label fields
        model_label (str): display name for the model
    """
    print("=" * 60)
    print(f"  {model_label}")
    print("=" * 60)

    correct_count = 0

    for i, case in enumerate(test_cases):
        raw_response = generate(model, case["prompt"])
        predicted_letter = extract_letter(raw_response)
        is_correct = predicted_letter == case["correct"]

        if is_correct:
            correct_count += 1
            verdict = "✅ CORRECT"
        else:
            verdict = "❌ WRONG  "

        print(f"\n  Q{i+1}: {case['question']}")
        print(f"       Choices  : {case['choices']}")
        print(f"       Expected : {case['correct_label']}")
        print(f"       Got      : {raw_response[:60]}")   # truncate verbose answers
        print(f"       Result   : {verdict}")

    print()
    print(f"  Score: {correct_count}/{len(test_cases)}")
    print("=" * 60)
    print()

    return correct_count


# ── Run Evaluation on Both Models ─────────────────────────────────────────────
ft_score   = evaluate_model(model,      test_cases, "FINE-TUNED MODEL (LoRA + Medical QA)")
base_score = evaluate_model(base_model, test_cases, "BASE MODEL (No Medical Fine-Tuning)")

# ── Final Comparison ───────────────────────────────────────────────────────────
print("=" * 60)
print("  FINAL COMPARISON")
print("=" * 60)
print(f"  Fine-tuned : {ft_score}/{len(test_cases)} correct")
print(f"  Base model : {base_score}/{len(test_cases)} correct")
print("=" * 60)

  FINE-TUNED MODEL (LoRA + Medical QA)

  Q1: ECG shows ST elevation in leads II, III, aVF. Diagnosis?
       Choices  : A: Anterior MI | B: Inferior MI | C: Lateral MI | D: Posterior MI
       Expected : B: Inferior MI
       Got      : A: Anterior MI
       Result   : ❌ WRONG  

  Q2: Which antibiotic is safe in pregnancy for UTI treatment?
       Choices  : A: Ciprofloxacin | B: Doxycycline | C: Nitrofurantoin | D: Tetracycline
       Expected : C: Nitrofurantoin
       Got      : C: Nitrofurantoin
       Result   : ✅ CORRECT

  Q3: High fever + neck stiffness + photophobia. Diagnosis?
       Choices  : A: Migraine | B: Meningitis | C: Encephalitis | D: Brain tumor
       Expected : B: Meningitis
       Got      : A: Migraine
       Result   : ❌ WRONG  

  Score: 1/3

  BASE MODEL (No Medical Fine-Tuning)

  Q1: ECG shows ST elevation in leads II, III, aVF. Diagnosis?
       Choices  : A: Anterior MI | B: Inferior MI | C: Lateral MI | D: Posterior MI
       Expected : B: Inferior MI

---
## 📚 Key Concepts Recap

| Concept | What It Means |
|---|---|
| **LoRA** | Inject small trainable matrices into frozen attention layers |
| **Rank (r)** | Bottleneck size of adapter matrices — controls expressivity vs. parameter count |
| **lora_alpha** | Scaling factor: effective influence = alpha/r |
| **target_modules** | Which weight matrices to apply LoRA to (q_proj, v_proj most impactful) |
| **Causal LM** | Labels = input_ids; model learns to predict every next token |
| **gradient_accumulation_steps** | Simulates large batch size without the memory cost |
| **Catastrophic Forgetting** | When fine-tuning destroys pre-trained knowledge — LoRA minimises this by freezing base weights |

---

## 🔗 References
- [LoRA Paper: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)
- [PEFT Library Documentation](https://huggingface.co/docs/peft)
- [TinyLlama Model Card](https://huggingface.co/TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T)
- [MedAlpaca Dataset](https://huggingface.co/datasets/medalpaca/medical_meadow_medqa)

---
*Built with ❤️ using HuggingFace PEFT | Google Colab T4 GPU*
